# RT-DETRv2 (R18VD), 640 px, one-class small-defect detection

This notebook uses the **same prepared data, label matching, seed-42 70/15/15 split, and overall/small/medium/large test sets** as the other baselines (`dfine_s_640_kaggle.ipynb`, `yolov8n-p2pan-imgsize640-fixed.ipynb`). Everything through the COCO-JSON export is copied from `dfine_s_640_kaggle.ipynb` unchanged.

**Framework choice, on purpose:** RT-DETRv2 is fine-tuned via Hugging Face `transformers` (`RTDetrV2ForObjectDetection`, checkpoint `PekingU/rtdetr_v2_r18vd`) rather than a cloned research repo, for two reasons:
- Ultralytics does not support RT-DETRv2 (only the original RT-DETR).
- `transformers.Trainer` has a native `EarlyStoppingCallback`, which is what implements the requested epoch-15 patience. The raw `lyuwenyu/RT-DETR` reference repo (what D-FINE's own codebase descends from) has no built-in early stopping, so it would have to be hand-rolled around repeated `train.py` calls.

Early stopping is driven by **validation loss** (`eval_loss`), not a custom mAP metric — every HF `...ForObjectDetection` model returns `.loss` for free when `labels` are passed, so this needs zero custom metric code during training, and `load_best_model_at_end=True` restores the best-checkpoint weights into the same `model` object used by the later evaluation cells. The authoritative mAP50 / mAP50-95 / Precision / Recall / FP-per-image numbers for the summary row are computed in a separate, simple post-training evaluation pass instead (same spirit as the other notebooks' size-wise eval cells), so a bug in a Trainer-internal metric callback can't silently corrupt the numbers that actually get reported.

**AMP is left off by default** (see `fp16` in the `TrainingArguments` cell). D-FINE — same DETR family — hit a NaN-box failure under AMP on this exact dataset. Start from full precision; only try `fp16=True` as a separate, deliberate experiment.

**Single GPU only, forced in code:** even on a T4 x2 accelerator, the imports cell pins `CUDA_VISIBLE_DEVICES=0`. Without that pin, `transformers.Trainer` auto-wraps the model in `torch.nn.DataParallel` whenever more than one GPU is visible (this notebook runs as a single process, no `accelerate launch`/`torchrun`). `DataParallel` can split stacked tensors like `pixel_values` across GPUs, but this notebook's `labels` are a plain Python list of per-image dicts — object-detection labels are variable-length, so they can't be a single stacked tensor — and `DataParallel` has no way to split a bare list. It silently duplicates the full list to every replica while still splitting `pixel_values` in half, so the denoising-query tensor (built from the un-split labels) and the regular query tensor (built from the correctly-split pixel values) disagree on batch size and the training step crashes inside RT-DETRv2's forward pass. Properly supporting multiple GPUs here would mean real distributed training with each GPU as its own process, which is a different notebook structure than the rest of this repo uses — pinning to one GPU sidesteps the problem entirely, at the cost of not using the second T4 if you picked T4 x2.

**Caveat:** unlike the other notebooks in this repo, this one hasn't been run end to end yet (no GPU available to verify it here). The dataset-prep cells are copied verbatim from `dfine_s_640_kaggle.ipynb` and are proven. The HF `Trainer`/`RTDetrV2ForObjectDetection` plumbing (cells under "Load RT-DETRv2" and "Evaluation") is new; the two riskiest assumptions in it — that `RTDetrImageProcessor` with `size={"height": 640, "width": 640}` does an exact, non-aspect-preserving resize (so `pixel_values` can be `torch.stack`-ed directly with no padding logic), and that `eval_strategy` (not the older `evaluation_strategy`) is the correct `TrainingArguments` kwarg — were both checked directly against the `transformers` source rather than guessed, so those should hold. What's a reasonable default rather than a verified value: the learning rate (`1e-4`) and warmup schedule aren't paper-derived the way D-FINE's batch-scaled LR was — watch the first few epochs' loss for instability, same failure class as D-FINE's own NaN history.

**Runtime:** `EPOCHS=100` with `PATIENCE=15`, on a single GPU, could take a while if the loss doesn't plateau early — worth factoring into how much of a Kaggle session (session cap is roughly 12 hours) this Save Version is likely to need.

Enable Kaggle Internet and a T4 GPU, attach **SmallDefectPreprocessing**, and run as a Save Version.

In [ ]:
import os

# Force single-GPU. If both T4s are visible, HF Trainer auto-wraps the model
# in torch.nn.DataParallel (this notebook is a single process, no
# accelerate launch/torchrun) — which cannot correctly scatter this
# notebook's list-based `labels` across devices, and crashes inside
# RTDetrV2's denoising-query concat with a batch-size mismatch. See the
# intro markdown cell for the full explanation. Must be set before `import
# torch` to take effect.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import json
import random
import shutil
import subprocess
import sys
import time
from collections import Counter, defaultdict
from pathlib import Path

import pandas as pd
import torch
from PIL import Image

assert Path("/kaggle/input").exists(), "This notebook must run on Kaggle."

In [ ]:
# transformers is left unpinned on purpose: RT-DETRv2 support landed in
# transformers relatively recently, and pinning an older version here would
# silently break this notebook by falling back to a version without it.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-U",
     "transformers", "accelerate", "torchmetrics", "pycocotools", "wandb", "pyyaml"],
    check=True,
)
print("Installed.")

In [ ]:
# Kaggle-specific safety check: pip-upgrading transformers/accelerate can, in
# rare cases, pull in a numpy or torch version that no longer matches
# Kaggle's preinstalled CUDA build (symptoms: numpy<->torch ABI errors, or
# CUDA silently becoming unavailable). Catch that here, immediately after
# install, rather than partway through a multi-hour training run.
import numpy as np

print("torch:", torch.__version__, "| torch's CUDA build:", torch.version.cuda)
print("numpy:", np.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Visible GPU count:", torch.cuda.device_count(), "(should be 1 — CUDA_VISIBLE_DEVICES pinned above)")
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# Canary for the numpy<->torch ABI mismatch class of error (e.g. "A module
# that was compiled using NumPy 1.x cannot be run in NumPy 2.x").
_smoke = torch.from_numpy(np.zeros(3, dtype=np.float32))
if torch.cuda.is_available():
    _smoke = (_smoke.cuda() + 1).cpu()
print("numpy <-> torch <-> CUDA smoke test passed.")

assert torch.cuda.is_available(), (
    "CUDA is not available after the pip install above. Either the GPU "
    "accelerator isn't enabled for this session (Settings > Accelerator > GPU), "
    "or upgrading transformers/accelerate pulled in a numpy or torch version that "
    "no longer matches Kaggle's preinstalled CUDA build. Do not continue on CPU — "
    "a 100-epoch DETR-family run would take days, not hours. If this fires, check "
    "the pip install cell's output above for a torch or numpy version change."
)
assert torch.cuda.device_count() == 1, (
    "More than one GPU is visible despite CUDA_VISIBLE_DEVICES being pinned in the "
    "imports cell — if you restarted the kernel without re-running that cell first, "
    "re-run from the top. Do not continue with >1 GPU visible; see the intro "
    "markdown for why DataParallel breaks this notebook's training step."
)

In [ ]:
# Run identity and fixed comparison settings.
RUN_NAME = "rtdetrv2_r18vd_imgsz640"
MODEL_LABEL = "RT-DETRv2-R18VD"
CHECKPOINT = "PekingU/rtdetr_v2_r18vd"
WANDB_PROJECT = "smallDefectDetection"

DATASET_NAMES = [
    "DAGM",
    "GC10-DET",
    "KolektorSDD2",
    "MPDD",
    "MTD",
    "Severstal",
    "VisA",
]
SIZE_BUCKETS = ["small", "medium", "large"]

SEED = 42
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

IMG_SIZE = 640
BATCH_SIZE = 8
EPOCHS = 100
PATIENCE = 15
WORKERS = 2
DEVICE = "cuda"  # already hard-asserted available one cell up — no silent CPU fallback

# Fixed evaluation policy shared with the other runs' FP-per-image code:
# match predictions to ground truth at conf=0.25, IoU=0.5. Precision/Recall
# here are read off that same fixed operating point, which is not
# necessarily how every other row in the comparison table derived its own
# Precision/Recall (some come from a framework validator's best-F1 point) —
# worth keeping in mind when comparing this row to the others.
CONF_THRESH = 0.25
MATCH_IOU = 0.5

BASE_DIR = Path("/kaggle/working")
KAGGLE_INPUT_ROOT = Path("/kaggle/input")
PREPARED_DATASET_DIR = BASE_DIR / "run_a_yolo_dataset"
COCO_ROOT = BASE_DIR / "rtdetrv2_defect_coco"
RUN_DIR = BASE_DIR / "rtdetrv2_runs" / RUN_NAME
FINAL_OUTPUT_DIR = BASE_DIR / "final_outputs" / RUN_NAME

random.seed(SEED)
os.environ["WANDB_PROJECT"] = WANDB_PROJECT

In [ ]:
def get_secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return os.environ.get(name)


wandb_key = get_secret("wandb_api_key")
if wandb_key:
    import wandb
    wandb.login(key=wandb_key)
    REPORT_TO = ["wandb"]
    print("W&B enabled. Project:", WANDB_PROJECT)
else:
    os.environ["WANDB_MODE"] = "disabled"
    REPORT_TO = []
    print("W&B secret not found; continuing with W&B disabled.")

In [ ]:
expected_datasets = set(DATASET_NAMES)

print("Searching for dataset root under:", KAGGLE_INPUT_ROOT)

valid_roots = []

for root, dirs, files in os.walk(KAGGLE_INPUT_ROOT):
    root_path = Path(root)
    dir_set = set(dirs)

    matches = expected_datasets.intersection(dir_set)

    if len(matches) >= 5:
        valid_roots.append(root_path)
        print("Found candidate:", root_path)
        print("Matches:", sorted(matches))

if not valid_roots:
    raise FileNotFoundError(
        "Could not find a folder containing the expected dataset folders: "
        f"{sorted(expected_datasets)}"
    )

hf_dataset_path = valid_roots[0]

print("\nUsing dataset root:", hf_dataset_path)
print("Available datasets:", sorted([p.name for p in hf_dataset_path.iterdir() if p.is_dir()]))

In [ ]:
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

samples = []
missing_count = 0

for dataset_name in DATASET_NAMES:
    for size_bucket in SIZE_BUCKETS:
        image_dir = hf_dataset_path / dataset_name / size_bucket / "images"
        label_dir = hf_dataset_path / dataset_name / size_bucket / "labels_yolo"

        if not image_dir.exists() or not label_dir.exists():
            print("Missing directory:", dataset_name, size_bucket)
            continue

        label_index = {
            label_path.stem: label_path
            for label_path in label_dir.glob("*.txt")
        }

        matched = 0
        bucket_missing = 0

        for image_path in image_dir.iterdir():
            if image_path.suffix.lower() not in IMAGE_EXTS:
                continue

            image_stem = image_path.stem
            base_stem = image_stem.removesuffix("_defect")

            possible_label_stems = [
                image_stem,
                image_stem.replace("_defect", "_bbs"),
                f"{base_stem}_bbs",
            ]

            label_path = next(
                (
                    label_index[stem]
                    for stem in possible_label_stems
                    if stem in label_index
                ),
                None,
            )

            if label_path is None:
                bucket_missing += 1
                continue

            samples.append({
                "image_path": image_path,
                "label_path": label_path,
                "dataset": dataset_name,
                "size": size_bucket,
                "stratum": f"{dataset_name}_{size_bucket}",
            })

            matched += 1

        missing_count += bucket_missing
        print(
            f"{dataset_name}/{size_bucket}: "
            f"{matched} matched, {bucket_missing} missing"
        )

print("\nTotal usable samples:", len(samples))
print("Total missing labels:", missing_count)

if not samples:
    raise RuntimeError("No image-label pairs were matched.")

In [ ]:
by_stratum = defaultdict(list)
for sample in samples:
    by_stratum[sample["stratum"]].append(sample)

train_samples, val_samples, test_samples = [], [], []
rng = random.Random(SEED)

for stratum, group in sorted(by_stratum.items()):
    group = list(group)
    rng.shuffle(group)
    n = len(group)
    n_train = int(n * TRAIN_RATIO)
    n_val = int(n * VAL_RATIO)

    train_samples.extend(group[:n_train])
    val_samples.extend(group[n_train:n_train + n_val])
    test_samples.extend(group[n_train + n_val:])

rng.shuffle(train_samples)
rng.shuffle(val_samples)
rng.shuffle(test_samples)


def count_by_size(split_samples, split_name):
    counts = Counter(s["size"] for s in split_samples)
    print(f"\n{split_name}")
    print("Total :", len(split_samples))
    print("Small :", counts["small"])
    print("Medium:", counts["medium"])
    print("Large :", counts["large"])


count_by_size(train_samples, "Train")
count_by_size(val_samples, "Val")
count_by_size(test_samples, "Test")
print("\nGrand total:", len(train_samples) + len(val_samples) + len(test_samples))

assert train_samples and val_samples and test_samples

In [ ]:
def reset_dir(path):
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)


reset_dir(PREPARED_DATASET_DIR)

splits_to_make = ["train", "val", "test", "test_small", "test_medium", "test_large"]

for split in splits_to_make:
    (PREPARED_DATASET_DIR / "images" / split).mkdir(parents=True, exist_ok=True)
    (PREPARED_DATASET_DIR / "labels" / split).mkdir(parents=True, exist_ok=True)


def rewrite_label_as_single_class(src_label_path, dst_label_path):
    new_lines = []
    with open(src_label_path, "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            coords = parts[1:5]
            new_lines.append("0 " + " ".join(coords))
    with open(dst_label_path, "w") as f:
        f.write("\n".join(new_lines))


def export_split(split_name, split_samples):
    for idx, sample in enumerate(split_samples):
        src_image = sample["image_path"]
        src_label = sample["label_path"]
        safe_name = f"{sample['dataset']}_{sample['size']}_{idx}_{src_image.name}"
        dst_image = PREPARED_DATASET_DIR / "images" / split_name / safe_name
        dst_label = PREPARED_DATASET_DIR / "labels" / split_name / f"{Path(safe_name).stem}.txt"
        shutil.copy2(src_image, dst_image)
        rewrite_label_as_single_class(src_label, dst_label)


export_split("train", train_samples)
export_split("val", val_samples)
export_split("test", test_samples)

test_small_samples = [s for s in test_samples if s["size"] == "small"]
test_medium_samples = [s for s in test_samples if s["size"] == "medium"]
test_large_samples = [s for s in test_samples if s["size"] == "large"]

export_split("test_small", test_small_samples)
export_split("test_medium", test_medium_samples)
export_split("test_large", test_large_samples)

print("YOLO dataset prepared.")
print("Test overall:", len(test_samples))
print("Test small:", len(test_small_samples))
print("Test medium:", len(test_medium_samples))
print("Test large:", len(test_large_samples))

In [ ]:
SPLITS = ("train", "val", "test", "test_small", "test_medium", "test_large")


def yolo_box_to_coco(parts, image_width, image_height):
    if len(parts) < 5:
        return None
    _, cx, cy, bw, bh = map(float, parts[:5])
    x1 = max(0.0, min((cx - bw / 2) * image_width, image_width))
    y1 = max(0.0, min((cy - bh / 2) * image_height, image_height))
    x2 = max(0.0, min((cx + bw / 2) * image_width, image_width))
    y2 = max(0.0, min((cy + bh / 2) * image_height, image_height))
    width, height = x2 - x1, y2 - y1
    if width <= 0 or height <= 0:
        return None
    return [round(x1, 4), round(y1, 4), round(width, 4), round(height, 4)]


def convert_split_to_coco(split_name):
    image_dir = PREPARED_DATASET_DIR / "images" / split_name
    label_dir = PREPARED_DATASET_DIR / "labels" / split_name
    images, annotations = [], []
    annotation_id = 1
    skipped_boxes = 0

    for image_id, image_path in enumerate(sorted(image_dir.iterdir()), start=1):
        if image_path.suffix.lower() not in IMAGE_EXTS:
            continue
        with Image.open(image_path) as image:
            width, height = image.size
        images.append({"id": image_id, "file_name": image_path.name, "width": width, "height": height})

        label_path = label_dir / f"{image_path.stem}.txt"
        for line in label_path.read_text().splitlines():
            box = yolo_box_to_coco(line.split(), width, height)
            if box is None:
                skipped_boxes += 1
                continue
            annotations.append({
                "id": annotation_id,
                "image_id": image_id,
                "category_id": 0,
                "bbox": box,
                "area": round(box[2] * box[3], 4),
                "iscrowd": 0,
            })
            annotation_id += 1

    coco = {
        "info": {"description": "One-class small-defect RT-DETRv2 dataset"},
        "licenses": [],
        "images": images,
        "annotations": annotations,
        "categories": [{"id": 0, "name": "defect", "supercategory": "defect"}],
    }
    annotation_dir = COCO_ROOT / "annotations"
    annotation_dir.mkdir(parents=True, exist_ok=True)
    output_path = annotation_dir / f"instances_{split_name}.json"
    output_path.write_text(json.dumps(coco, indent=2))
    return {"split": split_name, "images": len(images), "instances": len(annotations),
            "skipped_boxes": skipped_boxes, "annotation_file": str(output_path)}


conversion_rows = [convert_split_to_coco(split) for split in SPLITS]
conversion_df = pd.DataFrame(conversion_rows)
display(conversion_df)

# Same deterministic split logic/seed as dfine_s_640_kaggle.ipynb, so these
# counts should match it exactly. A mismatch means the attached dataset
# changed, not necessarily a bug — warn rather than hard-fail on it.
expected_images = {"train": 8858, "val": 1892, "test": 1920, "test_small": 462, "test_medium": 868, "test_large": 590}
for row in conversion_rows:
    if row["images"] != expected_images[row["split"]]:
        print(f"NOTE: {row['split']} has {row['images']} images, expected {expected_images[row['split']]}.")
    assert row["instances"] > 0 and row["skipped_boxes"] == 0, row

## Load RT-DETRv2 and wrap the COCO splits as an HF-style dataset

In [ ]:
from transformers import RTDetrImageProcessor, RTDetrV2ForObjectDetection

id2label = {0: "defect"}
label2id = {"defect": 0}

image_processor = RTDetrImageProcessor.from_pretrained(
    CHECKPOINT,
    size={"height": IMG_SIZE, "width": IMG_SIZE},
)

model = RTDetrV2ForObjectDetection.from_pretrained(
    CHECKPOINT,
    id2label=id2label,
    label2id=label2id,
    num_labels=len(id2label),
    ignore_mismatched_sizes=True,
)
model.to(DEVICE)
print("Loaded", CHECKPOINT, "on", DEVICE)

In [ ]:
class CocoDefectDataset(torch.utils.data.Dataset):
    def __init__(self, image_dir, ann_file, image_processor):
        self.image_dir = Path(image_dir)
        self.image_processor = image_processor
        coco = json.loads(Path(ann_file).read_text())
        self.images = {img["id"]: img for img in coco["images"]}
        self.image_ids = list(self.images.keys())
        self.annotations_by_image = defaultdict(list)
        for ann in coco["annotations"]:
            self.annotations_by_image[ann["image_id"]].append(ann)

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        image_id = self.image_ids[idx]
        image_info = self.images[image_id]
        image = Image.open(self.image_dir / image_info["file_name"]).convert("RGB")

        annotations = {
            "image_id": image_id,
            "annotations": self.annotations_by_image[image_id],
        }

        encoded = self.image_processor(images=image, annotations=annotations, return_tensors="pt")
        return {
            "pixel_values": encoded["pixel_values"][0],
            "labels": encoded["labels"][0],
        }


def collate_fn(batch):
    return {
        "pixel_values": torch.stack([item["pixel_values"] for item in batch]),
        "labels": [item["labels"] for item in batch],
    }


train_dataset = CocoDefectDataset(
    PREPARED_DATASET_DIR / "images" / "train",
    COCO_ROOT / "annotations" / "instances_train.json",
    image_processor,
)
val_dataset = CocoDefectDataset(
    PREPARED_DATASET_DIR / "images" / "val",
    COCO_ROOT / "annotations" / "instances_val.json",
    image_processor,
)

print("Train images:", len(train_dataset))
print("Val images:", len(val_dataset))

In [ ]:
def box_iou_xyxy(box1, box2):
    if len(box1) == 0 or len(box2) == 0:
        return torch.zeros((len(box1), len(box2)))
    x1 = torch.max(box1[:, 0].unsqueeze(1), box2[:, 0].unsqueeze(0))
    y1 = torch.max(box1[:, 1].unsqueeze(1), box2[:, 1].unsqueeze(0))
    x2 = torch.min(box1[:, 2].unsqueeze(1), box2[:, 2].unsqueeze(0))
    y2 = torch.min(box1[:, 3].unsqueeze(1), box2[:, 3].unsqueeze(0))

    inter = (x2 - x1).clamp(0) * (y2 - y1).clamp(0)
    area1 = (box1[:, 2] - box1[:, 0]) * (box1[:, 3] - box1[:, 1])
    area2 = (box2[:, 2] - box2[:, 0]) * (box2[:, 3] - box2[:, 1])
    union = area1.unsqueeze(1) + area2.unsqueeze(0) - inter
    return inter / union.clamp(min=1e-6)


def match_predictions(pred_boxes, pred_scores, gt_boxes, conf_thresh=CONF_THRESH, iou_thresh=MATCH_IOU):
    """Greedy match at a fixed confidence threshold. Returns (tp, fp, fn)."""
    keep = pred_scores >= conf_thresh
    pred_boxes = pred_boxes[keep]
    pred_scores = pred_scores[keep]

    if len(pred_boxes) == 0:
        return 0, 0, len(gt_boxes)
    if len(gt_boxes) == 0:
        return 0, len(pred_boxes), 0

    ious = box_iou_xyxy(pred_boxes, gt_boxes)
    matched_gt = set()
    tp = 0

    for pred_idx in torch.argsort(-pred_scores):
        best_gt = int(torch.argmax(ious[pred_idx]).item())
        best_iou = float(ious[pred_idx, best_gt].item())
        if best_iou >= iou_thresh and best_gt not in matched_gt:
            matched_gt.add(best_gt)
            tp += 1

    fp = len(pred_boxes) - tp
    fn = len(gt_boxes) - len(matched_gt)
    return tp, fp, fn

In [ ]:
from transformers import EarlyStoppingCallback, Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir=str(RUN_DIR),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    dataloader_num_workers=WORKERS,
    learning_rate=1e-4,
    weight_decay=1e-4,
    warmup_steps=300,
    lr_scheduler_type="cosine",
    # AMP off on purpose: D-FINE (same DETR family) hit a NaN-box failure
    # under AMP on this dataset. Try fp16=True as a separate experiment only.
    fp16=False,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_steps=50,
    seed=SEED,
    report_to=REPORT_TO,
    run_name=RUN_NAME,
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=collate_fn,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=PATIENCE)],
)

train_result = trainer.train()
print(train_result)

# load_best_model_at_end=True already reloaded the best checkpoint's weights
# into `model` in place (trainer.model IS model, same object) — these prints
# are just a visible record in the Save Version output of what actually won,
# since with early stopping the run may not reach epoch 100.
print("Best checkpoint:", trainer.state.best_model_checkpoint)
print("Best eval_loss:", trainer.state.best_metric)
print("Stopped after epoch:", trainer.state.epoch, "/ budget", EPOCHS)

## Evaluation: val, overall test, and small/medium/large test subsets

Runs a plain inference loop (`model(**inputs)` + `image_processor.post_process_object_detection`) rather than reusing anything Trainer-internal, so the reported numbers don't depend on getting HF's raw `EvalPrediction` tensor layout right.

In [ ]:
from torchmetrics.detection.mean_ap import MeanAveragePrecision


@torch.no_grad()
def run_split_evaluation(image_dir, ann_file, low_thresh=0.001):
    image_dir = Path(image_dir)
    coco = json.loads(Path(ann_file).read_text())
    images = {img["id"]: img for img in coco["images"]}
    anns_by_image = defaultdict(list)
    for ann in coco["annotations"]:
        anns_by_image[ann["image_id"]].append(ann)

    model.eval()
    map_metric = MeanAveragePrecision(box_format="xyxy", iou_type="bbox")

    total_tp = total_fp = total_fn = 0

    for image_id, image_info in images.items():
        image = Image.open(image_dir / image_info["file_name"]).convert("RGB")
        inputs = image_processor(images=image, return_tensors="pt").to(DEVICE)
        outputs = model(**inputs)

        result = image_processor.post_process_object_detection(
            outputs,
            threshold=low_thresh,
            target_sizes=torch.tensor([(image.height, image.width)]),
        )[0]

        pred_boxes = result["boxes"].cpu()
        pred_scores = result["scores"].cpu()

        gt = [ann["bbox"] for ann in anns_by_image[image_id]]
        gt_boxes_xywh = torch.tensor(gt, dtype=torch.float32) if gt else torch.zeros((0, 4))
        gt_boxes_xyxy = gt_boxes_xywh.clone()
        if len(gt_boxes_xyxy):
            gt_boxes_xyxy[:, 2] = gt_boxes_xywh[:, 0] + gt_boxes_xywh[:, 2]
            gt_boxes_xyxy[:, 3] = gt_boxes_xywh[:, 1] + gt_boxes_xywh[:, 3]

        map_metric.update(
            [{"boxes": pred_boxes, "scores": pred_scores, "labels": torch.zeros(len(pred_boxes), dtype=torch.int)}],
            [{"boxes": gt_boxes_xyxy, "labels": torch.zeros(len(gt_boxes_xyxy), dtype=torch.int)}],
        )

        tp, fp, fn = match_predictions(pred_boxes, pred_scores, gt_boxes_xyxy)
        total_tp += tp
        total_fp += fp
        total_fn += fn

    map_result = map_metric.compute()
    precision = total_tp / max(total_tp + total_fp, 1)
    recall = total_tp / max(total_tp + total_fn, 1)

    return {
        "mAP50": float(map_result["map_50"]),
        "mAP50_95": float(map_result["map"]),
        "Precision": precision,
        "Recall": recall,
        "FP_Per_Image": total_fp / max(len(images), 1),
        "num_images": len(images),
    }

In [ ]:
eval_sets = {
    "val": (PREPARED_DATASET_DIR / "images" / "val", COCO_ROOT / "annotations" / "instances_val.json"),
    "overall": (PREPARED_DATASET_DIR / "images" / "test", COCO_ROOT / "annotations" / "instances_test.json"),
    "small": (PREPARED_DATASET_DIR / "images" / "test_small", COCO_ROOT / "annotations" / "instances_test_small.json"),
    "medium": (PREPARED_DATASET_DIR / "images" / "test_medium", COCO_ROOT / "annotations" / "instances_test_medium.json"),
    "large": (PREPARED_DATASET_DIR / "images" / "test_large", COCO_ROOT / "annotations" / "instances_test_large.json"),
}

results = {}
for name, (image_dir, ann_file) in eval_sets.items():
    print("Evaluating:", name)
    results[name] = run_split_evaluation(image_dir, ann_file)
    print(results[name])

In [ ]:
def measure_inference_time_ms(image_paths, n_warmup=5, n_measure=50):
    model.eval()
    reps = (n_warmup + n_measure) // len(image_paths) + 1
    sample_paths = (image_paths * reps)[: n_warmup + n_measure]

    # Preprocess before starting the clock so this measures model forward
    # time only, matching how the YOLO-based rows report Inference_Time_ms
    # (their "inference" speed key, excluding preprocess/postprocess).
    prepared = []
    for path in sample_paths:
        image = Image.open(path).convert("RGB")
        inputs = image_processor(images=image, return_tensors="pt").to(DEVICE)
        prepared.append(inputs)

    with torch.no_grad():
        for inputs in prepared[:n_warmup]:
            model(**inputs)
    if DEVICE == "cuda":
        torch.cuda.synchronize()

    start = time.perf_counter()
    with torch.no_grad():
        for inputs in prepared[n_warmup:]:
            model(**inputs)
    if DEVICE == "cuda":
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - start

    return (elapsed / n_measure) * 1000


test_image_paths = sorted((PREPARED_DATASET_DIR / "images" / "test").iterdir())
inference_time_ms = measure_inference_time_ms(test_image_paths)
print(f"Inference time: {inference_time_ms:.2f} ms/image (model forward pass only, excludes preprocessing)")

In [ ]:
overall, small, medium, large = (results[k] for k in ("overall", "small", "medium", "large"))

summary_row = {
    "Experiment": RUN_NAME,
    "Model": MODEL_LABEL,
    "Batch": BATCH_SIZE,
    # Actual epochs trained, not the configured ceiling — with early
    # stopping these can differ, and the ceiling alone would misrepresent
    # how long this run actually took next to the other rows in the table.
    "Epochs": round(trainer.state.epoch),
    "mAP50": overall["mAP50"],
    "mAP50_95": overall["mAP50_95"],
    "Precision": overall["Precision"],
    "Recall": overall["Recall"],
    "mAP50_Small": small["mAP50"],
    "mAP50_Medium": medium["mAP50"],
    "mAP50_Large": large["mAP50"],
    "Recall_Small": small["Recall"],
    "Recall_Medium": medium["Recall"],
    "Recall_Large": large["Recall"],
    "Inference_Time_ms": inference_time_ms,
    "FP_Per_Image": overall["FP_Per_Image"],
    "Notes": (
        f"RT-DETRv2-R18VD via HF transformers, early stopping on eval_loss "
        f"(patience={PATIENCE}, epoch budget {EPOCHS}), fixed 640 split, P/R/FP at conf={CONF_THRESH}"
    ),
}

FINAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
summary_df = pd.DataFrame([summary_row])
summary_df.to_csv(FINAL_OUTPUT_DIR / "summary.csv", index=False)
pd.DataFrame([{"split": k, **v} for k, v in results.items()]).to_csv(
    FINAL_OUTPUT_DIR / "evaluation_metrics.csv", index=False
)
pd.DataFrame(conversion_rows).to_csv(FINAL_OUTPUT_DIR / "split_counts.csv", index=False)
trainer.save_model(str(FINAL_OUTPUT_DIR / "best_checkpoint"))
image_processor.save_pretrained(str(FINAL_OUTPUT_DIR / "best_checkpoint"))
shutil.copytree(COCO_ROOT / "annotations", FINAL_OUTPUT_DIR / "coco_annotations", dirs_exist_ok=True)

print("Saved final artifacts to:", FINAL_OUTPUT_DIR)
display(summary_df)